# 신용카드 이탈 데이터 기반 `education_id` 분석 및 예측 보고서

## 보고서 목적
본 노트북은 신용카드 고객 데이터에서 `education_id`의 분포와 특성을 분석하고,  
특히 `education_id == 7`로 표시된 미확인(Unknown) 집단이 기존 교육 수준 그룹과 어떤 관계를 가지는지 탐색하는 것을 목표로 합니다.

## 분석 범위
- 데이터 로드 및 기본 품질 점검
- `education_id` 기준 데이터 분리
- 스케일링 및 LDA 기반 저차원 시각화
- 학습/평가 데이터 분할 적정성 확인
- 다중분류 모델(Logistic Regression, Random Forest) 성능 비교
- 미확인 집단 해석을 위한 실무 관점 인사이트 정리

## 기대 효과
- 운영 데이터에서 미확인 범주가 실제 어떤 고객군에 가까운지 정성적으로 해석할 수 있습니다.
- 향후 결측 대체, 라벨 정제, 타깃 예측 모델 설계의 기초 자료로 활용할 수 있습니다.
- 팀 내 재현 가능한 분석 템플릿으로 재사용할 수 있습니다.


## 1. 분석 환경 및 라이브러리 준비

### 비즈니스 목적
실무 분석에서는 라이브러리 버전 차이, 시각화 설정, 출력 포맷 차이로 인해 같은 코드라도 결과 해석이 달라질 수 있습니다.  
따라서 초기에 공통 환경을 명확히 설정해두면 팀원 간 재현성과 유지보수성이 높아집니다.

### 분석 포인트
- 데이터 처리, 시각화, 모델링에 필요한 핵심 라이브러리만 선별합니다.
- 노트북 출력 품질을 위해 `display()` 기반 확인 방식을 사용할 준비를 합니다.
- 실무 배포 환경에서 외부 API 요청이 실패할 수 있으므로 예외 처리 기반 구조를 염두에 둡니다.


In [2]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

from IPython.display import display
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    precision_score,
    confusion_matrix,
    classification_report
)
from sklearn.utils.multiclass import type_of_target

warnings.filterwarnings("ignore")

# 왜 이 설정을 하는가:
# - 분석 결과가 셀마다 불필요하게 잘리지 않도록 출력 옵션을 확장합니다.
# - 협업 시 긴 컬럼명/요약표가 잘려 보이면 해석 오류가 생길 수 있기 때문입니다.
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 120)

# 왜 이 설정을 하는가:
# - 한글 제목/축 레이블이 깨지는 환경에서 시각화 해석이 어려워지므로 폰트 설정을 시도합니다.
# - 운영 서버/로컬 환경마다 사용 가능한 폰트가 다를 수 있어, 필요한 경우 사용자 환경에 맞게 수정해야 합니다.
plt.rcParams["axes.unicode_minus"] = False


## 2. 데이터 수집 함수 정의

### 비즈니스 목적
분석 대상 데이터를 API로 불러오는 함수를 별도 셀로 분리하면,  
향후 데이터 소스가 변경되더라도 로드 구간만 수정하여 전체 분석 흐름을 유지할 수 있습니다.

### 분석 포인트
- 데이터 로드 로직과 분석 로직을 분리해 유지보수성을 높입니다.
- API 응답 실패, 스키마 변경, 타입 불일치에 대비한 최소한의 방어 로직을 포함합니다.
- `X_y_split` 옵션을 통해 탐색과 모델링 양쪽에 유연하게 대응합니다.


In [3]:
def fetch_creditcard(X_y_split: bool = False) -> pd.DataFrame | tuple[pd.DataFrame, pd.Series]:
    # 왜 이 함수를 분리하는가:
    # - 데이터 수집 경로가 바뀌어도 이후 분석 셀 전체를 수정하지 않기 위해서입니다.
    # - 반복 분석 시 동일한 소스에 대해 일관된 로딩 방식을 보장할 수 있습니다.
    response = requests.get(
        "http://pipeline:8000/dataset/creditcard-churn",
        params={"X_y_split": X_y_split},
        timeout=30
    )

    # 왜 상태코드를 검증하는가:
    # - API 장애나 네트워크 문제 시 조용히 잘못된 데이터를 쓰는 것을 막기 위해서입니다.
    response.raise_for_status()
    payload = response.json()

    if X_y_split:
        # 유지보수 주의:
        # - API 응답 키 이름(x, y, index)이 바뀌면 이 구간에서 즉시 수정해야 합니다.
        X = pd.DataFrame(payload["x"], index=payload["index"])
        y = pd.Series(payload["y"], index=payload["index"], name="target")
        return X, y

    df = pd.DataFrame(payload["data"], index=payload["index"])
    return df


## 3. 원천 데이터 로드 및 1차 확인

### 비즈니스 목적
분석 시작 단계에서 데이터 크기와 샘플 형태를 빠르게 확인하면,  
이후 단계에서 발생하는 오류가 데이터 자체 문제인지 로직 문제인지 더 빠르게 분리할 수 있습니다.

### 분석 포인트
- 식별자 컬럼은 예측에 불필요한 경우가 많으므로 우선 제거 후보로 봅니다.
- 상위 몇 개 행과 shape를 함께 확인해 구조 이상 여부를 점검합니다.
- 출력은 `display()`를 사용해 팀원이 즉시 표 형태로 이해할 수 있도록 구성합니다.


In [5]:
# 왜 데이터를 먼저 전체 로드하는가:
# - EDA 단계에서는 컬럼 구성과 결측/분포를 함께 보는 것이 중요하기 때문입니다.
bank_df = fetch_creditcard()

# 왜 식별자 컬럼을 제거하는가:
# - 고객 고유 ID는 일반적으로 설명력이 아닌 식별 역할만 하므로 모델 성능을 왜곡할 수 있습니다.
# - 단, 추후 원본 데이터와 재조인해야 한다면 별도 보관 전략이 필요합니다.
bank_df = bank_df.drop("creditcard_churn_id", axis=1)

print(f"데이터 크기: {bank_df.shape[0]:,}행 × {bank_df.shape[1]:,}열")
display(bank_df.head())


ConnectionError: HTTPConnectionPool(host='pipeline', port=8000): Max retries exceeded with url: /dataset/creditcard-churn?X_y_split=False (Caused by NameResolutionError("HTTPConnection(host='pipeline', port=8000): Failed to resolve 'pipeline' ([Errno -5] No address associated with hostname)"))

## 4. 컬럼 구조 및 데이터 타입 점검

### 비즈니스 목적
실무에서는 모델보다 데이터 타입 문제로 장애가 발생하는 경우가 더 많습니다.  
숫자형이어야 할 컬럼이 문자열로 들어와 있거나, 예상치 못한 타입 변환이 있으면  
스케일링·통계검정·모델 학습이 모두 실패할 수 있습니다.

### 분석 포인트
- 컬럼 목록을 확인해 대상 변수와 설명 변수를 명확히 분리합니다.
- 데이터 타입과 결측 현황을 함께 점검하여 전처리 필요 영역을 식별합니다.
- 운영 배포 시 입력 스키마 검증 기준으로 활용할 수 있습니다.


In [ ]:
display(pd.DataFrame({
    "column_name": bank_df.columns,
    "dtype": bank_df.dtypes.astype(str).values,
    "null_count": bank_df.isnull().sum().values,
    "null_ratio": (bank_df.isnull().mean() * 100).round(2).values
}))

print("컬럼 목록:")
display(pd.DataFrame({"columns": bank_df.columns}))


## 5. 분석 대상 변수 분리

### 비즈니스 목적
이번 분석의 핵심은 `education_id`를 기준으로 다른 수치형 특성이 어떻게 분포하는지 확인하는 것입니다.  
따라서 타깃 역할을 하는 `education_id`와 입력 특성을 명확히 분리해야  
탐색·시각화·모델링 각 단계에서 혼선을 줄일 수 있습니다.

### 분석 포인트
- `education_id`를 제외한 나머지를 입력 변수 후보로 정의합니다.
- 타깃 분포를 먼저 확인해 클래스 불균형 여부를 가늠합니다.
- 향후 다중분류 평가 지표 선택 시 이 분포 확인이 중요합니다.


In [ ]:
# 왜 타깃과 피처를 분리하는가:
# - 분석 목적상 education_id는 설명 대상이며, 나머지 변수는 설명 변수이기 때문입니다.
features = [column for column in bank_df.columns if column != "education_id"]
features_df = bank_df[features].copy()
education_sr = bank_df["education_id"].copy()

print("입력 변수 개수:", len(features))
display(pd.DataFrame({"feature_name": features}))

print("education_id 분포:")
display(
    education_sr.value_counts(dropna=False)
    .sort_index()
    .rename_axis("education_id")
    .reset_index(name="count")
)


## 6. Known / Unknown 그룹 분리

### 비즈니스 목적
`education_id == 7`을 미확인 그룹으로 가정하고,  
정상적으로 라벨이 존재하는 고객군과 별도로 분리하면  
Unknown 집단이 실제로 어떤 기존 집단과 유사한지 탐색할 수 있습니다.

### 분석 포인트
- Known 집단은 모델 학습 및 기준 분포 해석에 사용합니다.
- Unknown 집단은 변환 후 투영하여 기존 집단과의 상대적 위치를 확인합니다.
- 이 단계는 라벨 정제/대체 전략 수립의 출발점입니다.


In [ ]:
# 왜 Unknown을 별도로 분리하는가:
# - 라벨이 불명확한 데이터를 그대로 학습시키면 모델이 기준 경계를 흐리게 학습할 수 있기 때문입니다.
# - 먼저 Known으로 기준 구조를 만든 뒤 Unknown을 사후 해석하는 접근이 더 실무적입니다.
UNKNOWN_EDUCATION_ID = 7

features_known_df = features_df[education_sr != UNKNOWN_EDUCATION_ID].copy()
features_unknown_df = features_df[education_sr == UNKNOWN_EDUCATION_ID].copy()
education_known_sr = education_sr[education_sr != UNKNOWN_EDUCATION_ID].copy()
education_unknown_sr = education_sr[education_sr == UNKNOWN_EDUCATION_ID].copy()

print("Known 데이터 크기:", features_known_df.shape)
print("Unknown 데이터 크기:", features_unknown_df.shape)

display(
    education_known_sr.value_counts()
    .sort_index()
    .rename_axis("education_id")
    .reset_index(name="count")
)


## 7. Known / Unknown 기본 통계 비교

### 비즈니스 목적
모델 학습 전에 두 집단의 기초 통계 차이를 보면  
Unknown 그룹이 특정 변수에서 구조적으로 다른지 빠르게 파악할 수 있습니다.  
이는 이후 LDA 시각화와 예측 결과 해석의 배경 정보가 됩니다.

### 분석 포인트
- 평균과 표준편차 차이가 큰 변수는 그룹 구분에 기여할 가능성이 높습니다.
- Unknown 집단이 전체적으로 편향된 특성을 보이는지 확인합니다.
- 차이가 크더라도 바로 인과로 해석하지 않고, 후속 검정과 모델 결과로 보완해야 합니다.


In [ ]:
known_summary_df = features_known_df.describe().T[["mean", "std", "min", "max"]].add_prefix("known_")
unknown_summary_df = features_unknown_df.describe().T[["mean", "std", "min", "max"]].add_prefix("unknown_")

comparison_summary_df = known_summary_df.join(unknown_summary_df, how="outer")
display(comparison_summary_df.head(20))


## 8. 스케일링 수행

### 비즈니스 목적
LDA와 로지스틱 회귀는 변수 스케일 차이에 영향을 받을 수 있으므로,  
특성 간 단위 차이를 줄여 모델이 특정 고스케일 변수만 과도하게 반영하지 않도록 합니다.

### 분석 포인트
- 스케일러는 반드시 Known 학습 기준으로 `fit`하고, Unknown에는 `transform`만 적용합니다.
- 데이터 누수를 방지하는 기본 원칙을 따릅니다.
- 배포 시에도 학습 시점 스케일러를 저장해 동일하게 재사용해야 합니다.


In [ ]:
scaler = StandardScaler()

# 왜 Known 데이터에만 fit 하는가:
# - Unknown 또는 테스트 데이터 정보를 평균/표준편차 계산에 사용하면 데이터 누수가 발생하기 때문입니다.
features_known_scaled = scaler.fit_transform(features_known_df)
features_unknown_scaled = scaler.transform(features_unknown_df)

# 유지보수 주의:
# - 실제 서비스에서는 scaler 객체를 파일로 저장(joblib/pickle 등)하여
#   학습/서빙 환경 간 동일한 변환을 보장해야 합니다.
print("Known 스케일링 결과 shape:", features_known_scaled.shape)
print("Unknown 스케일링 결과 shape:", features_unknown_scaled.shape)


## 9. LDA 차원 축소 수행

### 비즈니스 목적
다중 클래스 구조를 2차원으로 축약해 시각화하면,  
각 교육 수준 그룹이 서로 분리되는 정도와 Unknown 집단의 상대적 위치를 직관적으로 파악할 수 있습니다.

### 분석 포인트
- LDA는 클래스 분리 정보를 반영해 축을 만들기 때문에 단순 PCA보다 해석 목적에 유리할 수 있습니다.
- Known 집단으로 학습한 뒤 Unknown을 동일 공간에 투영합니다.
- 시각적 분리가 좋다고 해서 바로 예측 가능성이 높다고 단정하지는 않으며, 후속 모델 평가가 필요합니다.


In [ ]:
lda = LinearDiscriminantAnalysis(n_components=2)

# 왜 LDA를 사용하는가:
# - 이번 목적은 단순 분산 최대화보다 '교육 수준 클래스 간 분리 구조'를 보고 싶기 때문입니다.
known_lda = lda.fit_transform(features_known_scaled, education_known_sr)
unknown_lda = lda.transform(features_unknown_scaled)

known_lda_df = pd.DataFrame(known_lda, columns=["LD1", "LD2"], index=education_known_sr.index)
known_lda_df["education_id"] = education_known_sr.values

unknown_lda_df = pd.DataFrame(unknown_lda, columns=["LD1", "LD2"], index=education_unknown_sr.index)
unknown_lda_df["education_id"] = education_unknown_sr.values

display(known_lda_df.head())
display(unknown_lda_df.head())


## 10. Known 클래스별 LDA 시각화

### 비즈니스 목적
각 교육 수준 클래스가 LDA 공간에서 어느 정도 독립적인 군집을 이루는지 확인하면,  
모델이 실제로 구분 가능한 구조를 학습할 수 있을지 사전에 판단할 수 있습니다.

### 분석 포인트
- 클래스 간 중첩이 심하면 분류 성능이 제한될 수 있습니다.
- 특정 클래스만 명확히 떨어져 있다면 해당 클래스 위주로 잘 맞고 나머지는 어려울 수 있습니다.
- 축 해석 자체보다는 상대적 분리도와 군집 형태를 보는 것이 중요합니다.


In [ ]:
plt.figure(figsize=(10, 7))

for class_id in sorted(known_lda_df["education_id"].unique()):
    class_mask = known_lda_df["education_id"] == class_id
    plt.scatter(
        known_lda_df.loc[class_mask, "LD1"],
        known_lda_df.loc[class_mask, "LD2"],
        alpha=0.6,
        label=f"Education {class_id}"
    )

plt.title("Known 교육 수준 그룹의 LDA 분포")
plt.xlabel("LDA 축 1 (LD1)")
plt.ylabel("LDA 축 2 (LD2)")
plt.legend(title="education_id")
plt.grid(alpha=0.3)
plt.show()


## 11. Known vs Unknown 비교 시각화

### 비즈니스 목적
Unknown 집단을 같은 LDA 공간에 함께 표시하면,  
미확인 고객들이 특정 교육 수준 군집 근처에 모이는지 또는 별도의 분포를 보이는지 시각적으로 검토할 수 있습니다.

### 분석 포인트
- Unknown이 특정 Known 클래스 주변에 밀집하면 라벨 대체 후보로 검토할 수 있습니다.
- 넓게 퍼져 있으면 단일 클래스 대체보다 별도 처리 전략이 더 적절할 수 있습니다.
- 시각화는 의사결정 보조 수단이며, 최종 판단은 모델 확률·업무 규칙과 함께 봐야 합니다.


In [ ]:
plt.figure(figsize=(10, 7))

for class_id in sorted(known_lda_df["education_id"].unique()):
    class_mask = known_lda_df["education_id"] == class_id
    plt.scatter(
        known_lda_df.loc[class_mask, "LD1"],
        known_lda_df.loc[class_mask, "LD2"],
        alpha=0.35,
        label=f"Known {class_id}"
    )

plt.scatter(
    unknown_lda_df["LD1"],
    unknown_lda_df["LD2"],
    alpha=0.8,
    marker="x",
    label="Unknown (education_id=7)"
)

plt.title("Known 교육 수준과 Unknown 그룹의 LDA 비교")
plt.xlabel("LDA 축 1 (LD1)")
plt.ylabel("LDA 축 2 (LD2)")
plt.legend(title="그룹", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.grid(alpha=0.3)
plt.show()


## 12. 학습/평가 데이터 분리

### 비즈니스 목적
모델의 일반화 성능을 확인하려면 학습 데이터와 평가 데이터를 분리해야 합니다.  
특히 다중분류에서는 클래스 비율이 무너지지 않도록 계층적 분할이 중요합니다.

### 분석 포인트
- Known 데이터만 사용해 다중분류 모델을 학습합니다.
- `stratify`를 사용해 클래스 비율을 train/test에 유사하게 유지합니다.
- 재현 가능한 비교를 위해 `random_state`를 고정합니다.


In [ ]:
train_X, test_X, train_y, test_y = train_test_split(
    features_known_scaled,
    education_known_sr,
    test_size=0.2,
    random_state=42,
    stratify=education_known_sr
)

print("train_X shape:", train_X.shape)
print("test_X shape :", test_X.shape)
print("train_y shape:", train_y.shape)
print("test_y shape :", test_y.shape)


## 13. 분할 결과 검증 리포트

### 비즈니스 목적
학습/평가 셋의 클래스 분포가 크게 다르면 성능 비교가 왜곡될 수 있으므로,  
분할 직후 분포 검증을 별도 셀에서 명시적으로 수행합니다.

### 분석 포인트
- train/test 비율이 유사한지 확인합니다.
- 희소 클래스가 테스트 셋에서 지나치게 적어지지 않았는지 확인합니다.
- 이 결과는 성능 수치의 신뢰도를 판단하는 기초 자료입니다.


In [ ]:
def build_split_report(X_train, X_test, y_train, y_test) -> dict:
    # 왜 분할 리포트를 함수로 만드는가:
    # - 여러 모델/실험에서 동일 형식으로 검증하기 쉽고, 누락 없이 점검할 수 있기 때문입니다.
    return {
        "X_train_shape": X_train.shape,
        "X_test_shape": X_test.shape,
        "y_train_shape": y_train.shape,
        "y_test_shape": y_test.shape,
        "train_distribution": y_train.value_counts().sort_index(),
        "test_distribution": y_test.value_counts().sort_index(),
        "train_ratio": y_train.value_counts(normalize=True).sort_index().round(4),
        "test_ratio": y_test.value_counts(normalize=True).sort_index().round(4),
    }

split_report = build_split_report(train_X, test_X, train_y, test_y)

print("Train / Test Split Summary")
print("=" * 60)
print("X_train_shape:", split_report["X_train_shape"])
print("X_test_shape :", split_report["X_test_shape"])
print("y_train_shape:", split_report["y_train_shape"])
print("y_test_shape :", split_report["y_test_shape"])

display(
    pd.concat(
        [
            split_report["train_distribution"].rename("train_count"),
            split_report["test_distribution"].rename("test_count"),
            split_report["train_ratio"].rename("train_ratio"),
            split_report["test_ratio"].rename("test_ratio"),
        ],
        axis=1
    ).reset_index(names="education_id")
)


## 14. 공통 평가 함수 정의

### 비즈니스 목적
여러 모델을 비교할 때 평가 기준이 매번 달라지면 해석이 어려워집니다.  
공통 평가 함수를 두면 팀 내 실험 결과를 일관된 포맷으로 축적할 수 있습니다.

### 분석 포인트
- 정확도뿐 아니라 recall, precision, F1, ROC-AUC, PR-AUC를 함께 확인합니다.
- 다중분류에 맞춰 `macro` 평균 기준을 사용합니다.
- 함수 내부에서 타깃 유형을 자동 판단하여 재사용성을 높입니다.


In [ ]:
def evaluate_classification(y_true, y_pred, y_proba, classes=None, average="macro", dataset_name="Dataset") -> pd.DataFrame:
    target_type = type_of_target(y_true)
    y_proba = np.asarray(y_proba)

    result = {
        "dataset": dataset_name,
        "target_type": target_type,
        "accuracy": accuracy_score(y_true, y_pred),
    }

    if target_type == "binary":
        # 왜 binary / multiclass를 나누는가:
        # - ROC-AUC와 PR-AUC 계산 방식이 타깃 구조에 따라 달라지기 때문입니다.
        result["recall"] = recall_score(y_true, y_pred)
        result["f1"] = f1_score(y_true, y_pred)

        if y_proba.ndim == 2 and y_proba.shape[1] == 2:
            y_score = y_proba[:, 1]
        elif y_proba.ndim == 1:
            y_score = y_proba
        else:
            raise ValueError(f"Binary classification에서 y_proba shape가 부적절합니다: {y_proba.shape}")

        result["roc_auc"] = roc_auc_score(y_true, y_score)
        result["pr_auc"] = average_precision_score(y_true, y_score)
        result["precision"] = precision_score(y_true, y_pred)

    elif target_type == "multiclass":
        result["recall"] = recall_score(y_true, y_pred, average=average)
        result["f1"] = f1_score(y_true, y_pred, average=average)
        result["precision"] = precision_score(y_true, y_pred, average=average)

        if y_proba.ndim != 2:
            raise ValueError(f"Multiclass classification에서는 y_proba가 2차원이어야 합니다: {y_proba.shape}")

        result["roc_auc"] = roc_auc_score(
            y_true,
            y_proba,
            multi_class="ovr",
            average=average
        )

        if classes is None:
            classes = np.unique(y_true)

        y_true_bin = label_binarize(y_true, classes=classes)
        result["pr_auc"] = average_precision_score(
            y_true_bin,
            y_proba,
            average=average
        )
    else:
        raise ValueError(f"지원하지 않는 target type입니다: {target_type}")

    return pd.DataFrame([result])


def display_model_result(y_true, y_pred, y_proba, classes, dataset_name, model_name):
    result_df = evaluate_classification(
        y_true=y_true,
        y_pred=y_pred,
        y_proba=y_proba,
        classes=classes,
        dataset_name=dataset_name
    )
    result_df.insert(0, "model", model_name)
    display(result_df)
    return result_df


## 15. Logistic Regression 학습

### 비즈니스 목적
로지스틱 회귀는 해석 가능성과 기준선(baseline) 확보 측면에서 실무에서 자주 사용하는 모델입니다.  
복잡한 모델보다 먼저 간단한 선형 기반 모델을 적용해 분류 가능성을 점검합니다.

### 분석 포인트
- 선형 분리 가능성이 어느 정도 있는지 확인하는 기준 모델입니다.
- 스케일링된 데이터와 궁합이 좋습니다.
- 결과가 기대보다 낮더라도 이후 복잡한 모델과의 개선 폭을 해석하는 기준이 됩니다.


In [ ]:
lr_model = LogisticRegression(max_iter=1000, random_state=42)

# 왜 max_iter를 늘리는가:
# - 다중분류와 다차원 특성 조합에서 기본 반복 수로는 수렴 경고가 날 수 있기 때문입니다.
lr_model.fit(train_X, train_y)

lr_train_pred = lr_model.predict(train_X)
lr_test_pred = lr_model.predict(test_X)

lr_train_proba = lr_model.predict_proba(train_X)
lr_test_proba = lr_model.predict_proba(test_X)


## 16. Logistic Regression 성능 평가

### 비즈니스 목적
학습 셋과 테스트 셋 성능을 함께 보면 과적합 여부를 빠르게 판단할 수 있습니다.  
또한 다중분류 문제에서는 단순 정확도보다 macro 기준의 정밀도·재현율·F1을 함께 보는 것이 중요합니다.

### 분석 포인트
- Train과 Test 차이가 과도하면 일반화 성능 저하를 의심합니다.
- ROC-AUC와 PR-AUC는 클래스 확률 품질까지 함께 판단하는 데 도움이 됩니다.
- 이후 Random Forest와 상대 비교해 어떤 구조가 더 잘 맞는지 확인합니다.


In [ ]:
lr_train_result_df = display_model_result(
    y_true=train_y,
    y_pred=lr_train_pred,
    y_proba=lr_train_proba,
    classes=lr_model.classes_,
    dataset_name="Train",
    model_name="LogisticRegression"
)

lr_test_result_df = display_model_result(
    y_true=test_y,
    y_pred=lr_test_pred,
    y_proba=lr_test_proba,
    classes=lr_model.classes_,
    dataset_name="Test",
    model_name="LogisticRegression"
)

print("Logistic Regression - Test Classification Report")
print(classification_report(test_y, lr_test_pred))


## 17. Logistic Regression 혼동행렬 확인

### 비즈니스 목적
어떤 교육 수준끼리 특히 자주 혼동되는지 확인하면,  
모델 개선 방향뿐 아니라 데이터 라벨 체계 자체의 중첩 가능성도 검토할 수 있습니다.

### 분석 포인트
- 혼동이 집중되는 클래스 쌍을 확인합니다.
- 실제로 비즈니스 상 구분이 어려운 구간인지 검토할 수 있습니다.
- 라벨 병합, 재정의, 추가 변수 확보 필요성을 판단하는 근거가 됩니다.


In [ ]:
lr_cm = pd.DataFrame(
    confusion_matrix(test_y, lr_test_pred, labels=lr_model.classes_),
    index=[f"Actual_{label}" for label in lr_model.classes_],
    columns=[f"Pred_{label}" for label in lr_model.classes_]
)

display(lr_cm)


## 18. Random Forest 학습

### 비즈니스 목적
트리 기반 모델은 변수 간 비선형 관계와 상호작용을 포착하는 데 유리하므로,  
선형 모델 대비 성능 개선 여지가 있는지 확인하기 위해 비교 모델로 사용합니다.

### 분석 포인트
- 로지스틱 회귀보다 복잡한 패턴을 학습할 수 있습니다.
- 현재는 해석 가능한 수준의 얕은 깊이(`max_depth=3`)로 제한해 과적합을 억제합니다.
- 이후 필요 시 깊이, 트리 수, 최소 샘플 수를 튜닝할 수 있습니다.


In [ ]:
rf_model = RandomForestClassifier(
    random_state=42,
    max_depth=3,
    n_estimators=200
)

# 왜 깊이를 제한하는가:
# - 지나치게 깊은 트리는 학습 데이터에 과적합될 가능성이 높기 때문입니다.
# - 현재 단계는 최고 성능보다 안정적인 비교 기준 확보에 초점을 둡니다.
rf_model.fit(train_X, train_y)

rf_train_pred = rf_model.predict(train_X)
rf_test_pred = rf_model.predict(test_X)

rf_train_proba = rf_model.predict_proba(train_X)
rf_test_proba = rf_model.predict_proba(test_X)


## 19. Random Forest 성능 평가

### 비즈니스 목적
기준선 모델과 트리 기반 모델을 동일 기준으로 평가해  
현재 문제 구조에 더 적합한 접근이 무엇인지 판단합니다.

### 분석 포인트
- Test 성능이 개선되면 비선형 패턴의 존재 가능성을 시사합니다.
- Train만 높고 Test가 낮으면 모델 복잡도가 과한 것일 수 있습니다.
- 실무 적용 시 해석 가능성과 성능 개선 폭을 함께 고려해야 합니다.


In [ ]:
rf_train_result_df = display_model_result(
    y_true=train_y,
    y_pred=rf_train_pred,
    y_proba=rf_train_proba,
    classes=rf_model.classes_,
    dataset_name="Train",
    model_name="RandomForest"
)

rf_test_result_df = display_model_result(
    y_true=test_y,
    y_pred=rf_test_pred,
    y_proba=rf_test_proba,
    classes=rf_model.classes_,
    dataset_name="Test",
    model_name="RandomForest"
)

print("Random Forest - Test Classification Report")
print(classification_report(test_y, rf_test_pred))


## 20. Random Forest 변수 중요도 확인

### 비즈니스 목적
트리 기반 모델이 어떤 변수를 상대적으로 더 많이 활용했는지 확인하면,  
향후 데이터 수집 우선순위나 라벨 정제 전략 수립에 도움을 줄 수 있습니다.

### 분석 포인트
- 중요도가 높은 변수는 교육 수준 구분에 상대적으로 기여할 가능성이 있습니다.
- 중요도는 인과관계가 아니라 모델 내부 분기 기준의 상대적 기여도입니다.
- 고상관 변수끼리는 중요도가 분산될 수 있어 단독 해석에 주의해야 합니다.


In [ ]:
feature_importance_df = pd.DataFrame({
    "feature": features_known_df.columns,
    "importance": rf_model.feature_importances_
}).sort_values("importance", ascending=False)

display(feature_importance_df.head(15))

plt.figure(figsize=(10, 6))
plt.barh(feature_importance_df["feature"].head(15)[::-1], feature_importance_df["importance"].head(15)[::-1])
plt.title("Random Forest 기준 상위 15개 변수 중요도")
plt.xlabel("중요도")
plt.ylabel("변수명")
plt.grid(axis="x", alpha=0.3)
plt.show()


## 21. 모델 성능 비교 요약

### 비즈니스 목적
여러 성능 지표를 한 표로 모아두면,  
회의나 보고에서 어떤 모델을 기준안으로 채택할지 빠르게 논의할 수 있습니다.

### 분석 포인트
- Test 기준으로 모델을 우선 비교합니다.
- Accuracy만 보지 않고 precision, recall, F1, ROC-AUC, PR-AUC를 함께 판단합니다.
- 성능 차이가 작다면 더 단순하고 설명 가능한 모델이 실무적으로 유리할 수 있습니다.


In [ ]:
model_comparison_df = pd.concat(
    [
        lr_train_result_df,
        lr_test_result_df,
        rf_train_result_df,
        rf_test_result_df
    ],
    ignore_index=True
)

display(model_comparison_df.sort_values(["dataset", "accuracy"], ascending=[True, False]))


## 22. Unknown 그룹 예측 확률 확인

### 비즈니스 목적
Unknown 그룹을 단순히 하나의 클래스로 남겨둘지,  
아니면 가장 가능성이 높은 기존 교육 수준으로 보조 라벨링할지 판단하기 위해  
모델 예측 확률을 확인합니다.

### 분석 포인트
- 확률이 특정 클래스에 높게 몰리면 대체 라벨 후보로 검토할 수 있습니다.
- 확률이 전반적으로 낮고 분산되면 별도 Unknown 유지 전략이 더 안전합니다.
- 자동 대체는 업무 규칙과 검증 절차 없이 바로 적용하지 않는 것이 바람직합니다.


In [ ]:
unknown_pred_proba_df = pd.DataFrame(
    lr_model.predict_proba(features_unknown_scaled),
    columns=[f"class_{class_id}_proba" for class_id in lr_model.classes_],
    index=features_unknown_df.index
)

unknown_pred_summary_df = pd.DataFrame({
    "predicted_class": lr_model.predict(features_unknown_scaled),
    "max_probability": unknown_pred_proba_df.max(axis=1)
}, index=features_unknown_df.index).join(unknown_pred_proba_df)

display(unknown_pred_summary_df.head(10))

display(
    unknown_pred_summary_df["predicted_class"]
    .value_counts()
    .sort_index()
    .rename_axis("predicted_class")
    .reset_index(name="count")
)


## 23. 결과 저장

### 비즈니스 목적
실무에서는 노트북 내 출력만으로 끝나지 않고,  
후속 회의 자료나 배치 파이프라인 입력용 파일로 결과를 저장하는 경우가 많습니다.  
따라서 핵심 산출물을 파일로 내보내는 셀을 별도로 분리합니다.

### 분석 포인트
- 비교표, 변수 중요도, Unknown 예측 결과를 저장 대상으로 선정합니다.
- 경로는 환경별로 달라질 수 있으므로 상대경로/절대경로 정책을 팀 내에서 통일해야 합니다.
- 운영 환경에서는 덮어쓰기 정책과 권한 문제를 사전에 확인해야 합니다.


In [ ]:
from pathlib import Path

output_dir = Path("./output")
output_dir.mkdir(parents=True, exist_ok=True)

# 왜 저장 셀을 별도로 두는가:
# - 분석 검토 단계에서는 실행하지 않고, 확정 산출물만 선택적으로 저장할 수 있기 때문입니다.
model_comparison_df.to_csv(output_dir / "model_comparison.csv", index=False, encoding="utf-8-sig")
feature_importance_df.to_csv(output_dir / "random_forest_feature_importance.csv", index=False, encoding="utf-8-sig")
unknown_pred_summary_df.to_csv(output_dir / "unknown_education_prediction_summary.csv", index=True, encoding="utf-8-sig")

print(f"저장 완료: {output_dir.resolve()}")


## 24. 실무 해석 요약

### 비즈니스 목적
노트북의 마지막에는 분석 결과를 실무 의사결정 문장으로 정리해야  
팀원이 코드 세부 내용을 몰라도 바로 액션 아이템을 도출할 수 있습니다.

### 분석 포인트
- LDA 시각화로 Known/Unknown의 상대 위치를 해석합니다.
- 모델 비교 결과로 분류 가능성과 과적합 여부를 함께 판단합니다.
- Unknown 자동 대체 여부는 예측 확률 분포와 업무 리스크를 고려해 결정합니다.

### 실무 제안
1. Unknown(`education_id=7`)이 특정 클래스 주변에 일관되게 모인다면,  
   수동 검증 표본을 추출해 라벨 정제 가능성을 검토합니다.
2. 모델 성능이 충분하지 않다면,  
   현재 변수만으로는 교육 수준 구분 한계가 있을 수 있으므로 추가 고객 속성 확보를 검토합니다.
3. 배포 단계에서는 반드시 다음을 점검해야 합니다.
   - 입력 컬럼 순서와 데이터 타입 고정
   - 학습 시점 스케일러 재사용
   - Unknown 자동 라벨링 임계값 설정
   - API 경로 및 저장 경로 환경 분리
